# Machine Learning Models
In this notebook, we train and compare three different machine learning models to detect fraud: Logistic Regression, Random Forest, and XGBoost.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve
)

# Load data
df = pd.read_csv('../data/processed.csv')

# Setup data
X = df.drop(columns=['isFraud', 'step'])
y = df['isFraud']

# Split 80/20 with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}, Fraud cases: {y_train.sum()}")
print(f"Testing set shape: {X_test.shape}, Fraud cases: {y_test.sum()}")

## 1. Train and Evaluate Models

**How do they differ?**
- **Logistic Regression:** The simplest model. It tries to draw a straight line (or plane) through the data to separate fraud from non-fraud. It looks at all variables linearly.
- **Random Forest:** An "ensemble" model. It builds hundreds of Decision Trees (which are basically flowcharts asking yes/no questions like "Is errorBalance > 0?"). Each tree votes on whether a transaction is fraud, and the majority wins.
- **XGBoost:** Another "ensemble" of trees, but much smarter. Instead of building all trees independently, it builds them one by one. Each new tree specifically focuses on correcting the mistakes made by the previous trees.

In [ ]:
# Dictionary to store probabilities for plotting later
probs_dict = {}
metrics_list = []

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    print(f"\n{'='*40}")
    print(f" {name} ")
    print(f"{'='*40}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}")
    
    metrics_list.append({
        'Model': name,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc
    })
    return y_prob

### Model 1: Logistic Regression ###
log_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
log_reg.fit(X_train, y_train)
probs_dict['Logistic Regression'] = evaluate_model('Logistic Regression', log_reg, X_test, y_test)

### Model 2: Random Forest ###
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
probs_dict['Random Forest'] = evaluate_model('Random Forest', rf, X_test, y_test)

### Model 3: XGBoost ###
# Calculate the ratio of negative (non-fraud) vs positive (fraud) cases for scale_pos_weight
num_non_fraud = (y_train == 0).sum()
num_fraud = (y_train == 1).sum()
scale_weight = num_non_fraud / num_fraud

xgb = XGBClassifier(scale_pos_weight=scale_weight, eval_metric='aucpr', random_state=42)
xgb.fit(X_train, y_train)
probs_dict['XGBoost'] = evaluate_model('XGBoost', xgb, X_test, y_test)

## 2. Metrics Comparison Table

In [ ]:
metrics_df = pd.DataFrame(metrics_list).set_index('Model')
display(metrics_df.style.highlight_max(color='lightgreen', axis=0))

## 3. Precision-Recall Curves
The PR-Curve plots Precision against Recall at all possible thresholds. A perfect model would hug the top-right corner. The closer a curve is to the top right (higher PR-AUC), the better the model is at catching fraud without accidentally blocking normal transactions.

In [ ]:
plt.figure(figsize=(10, 7))
for name, y_prob in probs_dict.items():
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    plt.plot(recall, precision, lw=2, label=f"{name} (PR-AUC = {pr_auc:.4f})")
    
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Feature Importance
Tree-based models like Random Forest and XGBoost can tell us exactly which variables they relied on the most to make their decisions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest Feature Importance
rf_importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
rf_importances.plot(kind='barh', ax=axes[0], color='skyblue')
axes[0].set_title('Random Forest Feature Importance')
axes[0].set_xlabel('Importance Score')

# XGBoost Feature Importance
xgb_importances = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=True)
xgb_importances.plot(kind='barh', ax=axes[1], color='lightgreen')
axes[1].set_title('XGBoost Feature Importance')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

## Conclusion: Which model is best and why?

**XGBoost** is expected to be the top performer, though **Random Forest** is typically a close second. Logistic Regression generally falls far behind.

**Why?**
1. **Non-linear relationships:** Logistic Regression tries to draw a straight line to separate fraud from non-fraud. But fraud patterns are complex (e.g., "If it's past 1 AM AND the amount is very high AND the balance goes exactly to zero, then it's fraud"). Decision trees (like Random Forest and XGBoost) can easily learn these complex "If-This-Then-That" rules.
2. **Learning from mistakes:** Random Forest builds all its trees independently at the same time. XGBoost builds trees sequentially, where every new tree acts as a critic—specifically looking at the mistakes made by the previous trees and forcing the model to learn from them. This makes XGBoost incredibly powerful and highly accurate for rare events like fraud detection.
3. **Feature Importance:** By looking at the charts above, you will see that the new features we engineered (like `errorBalanceOrig`) are the most heavily relied upon by the winning models. This proves that having good data features is just as important as having a good AI algorithm!